In [24]:
pip install neo4j rdflib

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
from zipfile import ZipFile
import requests
from io import BytesIO
from neo4j import GraphDatabase, basic_auth
import json
from rdflib import Graph

### Functions to interact with Neo4j

In [30]:
driver = GraphDatabase.driver(
  "neo4j://127.0.0.1:7687",
  auth=basic_auth("neo4j", "eduardo1989"))

def store_onto(driver, dbname, onto_as_string):
  onto_import_query = '''
  call n10s.rdf.import.inline($payload,'Turtle')
  '''

  with driver.session(database=dbname) as session:
    results = session.execute_write(
      lambda tx: tx.run(onto_import_query, payload=onto_as_string).data())
    for record in results:
      print(record)

def store_onto_delta(driver, dbname, deleted, added):
  onto_add_query = '''
  call n10s.rdf.import.inline($payload,'Turtle')
  '''
  onto_delete_query = '''
  call n10s.rdf.delete.inline($payload,'Turtle')
  '''
  with driver.session(database=dbname) as session:
    results = session.execute_write(
      lambda tx: tx.run(onto_add_query, payload=added).data())
    for record in results:
      print(record)
    results = session.execute_write(
      lambda tx: tx.run(onto_delete_query, payload=deleted).data())
    for record in results:
      print(record)

### Get the current version of the ontology from the dev tool and store it in Neo4j

In [31]:
meu_project_id = "9ab43003-23e1-42f7-a996-a2110565ccea" 

meu_session_id = "AA2180454CD255D1A8D91F41760010D3"

web_protege_url = f"https://webprotege.stanford.edu/download?project={meu_project_id}&format=ttl"
web_protege_cookies = {'JSESSIONID': meu_session_id}

print("URL configurada:", web_protege_url) 

URL configurada: https://webprotege.stanford.edu/download?project=9ab43003-23e1-42f7-a996-a2110565ccea&format=ttl


In [32]:
with ZipFile(BytesIO(requests.get(web_protege_url, cookies = web_protege_cookies).content)) as zp:
  for filename in zp.namelist():
    print(filename)
    store_onto(driver, "neo4j", zp.read(filename).decode("utf-8"))

grupos-pet-ontologies-ttl-REVISION-HEAD/grupos-pet.ttl
{'terminationStatus': 'OK', 'triplesLoaded': 160, 'triplesParsed': 160, 'namespaces': None, 'extraInfo': '', 'callParams': {'singleTx': True}}


### Get the current ontology in Neo4j and load it into an RdfLib Graph object

In [62]:
neo4j_rdf_endpoint = 'http://127.0.0.1:7474/rdf/neo4j/cypher'
query_get_ontology = 'match (r:Resource) return r union match (:Resource)-[r]-() return r'

payload = { 'cypher' : query_get_ontology , 'format' : 'Turtle' }

response = requests.post(neo4j_rdf_endpoint, auth=('neo4j', 'eduardo1989'), data = json.dumps(payload))
response.raise_for_status()  # raise an error on unsuccessful status codes

current = Graph()
current.parse(data=response.text, format='turtle')


<Graph identifier=Ne17cefa7d83e41d0b507049758e8c3e1 (<class 'rdflib.graph.Graph'>)>

In [63]:
print(current.serialize())

@prefix n4sch: <neo4j://graph.schema#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<http://webprotege.stanford.edu/R9WOQFMALIkCbr2tAcwPuW> a n4sch:DatatypeProperty ;
    n4sch:comment "Propriedade geral de Nome para ser usada na Ontologia." ;
    n4sch:label "hasName" ;
    n4sch:range xsd:string ;
    n4sch:subPropertyOf owl:topDataProperty .

owl:backwardCompatibleWith n4sch:range xsd:string .

owl:topObjectProperty n4sch:comment """Please provide the explanatory comments about the properties always in a one-to-one manner to make interpretation easier.

Of course, if the property has no cardinality restrictions, we can instantiate it multiple times. For example, hasCampus refers to “a university has a campus.” Logically, it can be instantiated zero to many times:

UTFPR hasCampus Curitiba
UTFPR hasCampus Cornélio Procópio
UTFPR hasCampus Dois Vizinhos""",
        """Por favor, forneça os comentários explicativos sobre as propr

### Get the current ontology in WebProtege and load it into an RdfLib Graph object

In [64]:
new = Graph()
with ZipFile(BytesIO(requests.get(web_protege_url, cookies = web_protege_cookies).content)) as zp:
  for filename in zp.namelist():
    print(filename)
    new.parse(data=zp.read(filename).decode("utf-8"), format='turtle')

grupos-pet-ontologies-ttl-REVISION-HEAD/grupos-pet.ttl


### Do delta analysis and persist as needed

In [65]:
added = new - current
deleted = current - new
print("ADDED TRIPLES")
print(added.serialize(format="turtle"))
print("DELETED TRIPLES")
print(deleted.serialize(format="turtle"))

ADDED TRIPLES
@prefix ns1: <https://github.com/peteco-utfpr/grupos-pet/> .
@prefix ns2: <http://webprotege.stanford.edu/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ns2:R9WOQFMALIkCbr2tAcwPuW a owl:DatatypeProperty ;
    rdfs:label "hasName" ;
    rdfs:comment "Propriedade geral de Nome para ser usada na Ontologia."^^xsd:string ;
    rdfs:range xsd:string ;
    rdfs:subPropertyOf owl:topDataProperty .

owl:backwardCompatibleWith rdfs:range xsd:string .

owl:topObjectProperty rdfs:comment """Please provide the explanatory comments about the properties always in a one-to-one manner to make interpretation easier.

Of course, if the property has no cardinality restrictions, we can instantiate it multiple times. For example, hasCampus refers to “a university has a campus.” Logically, it can be instantiated zero to many times:

UTFPR hasCampus Curitiba
UTFPR hasCampus Cornélio P

In [66]:
store_onto_delta(driver, "neo4j", deleted.serialize(format="turtle"), added.serialize(format="turtle"))

{'terminationStatus': 'OK', 'triplesLoaded': 177, 'triplesParsed': 177, 'namespaces': None, 'extraInfo': '', 'callParams': {'singleTx': True}}
{'terminationStatus': 'OK', 'triplesDeleted': 151, 'namespaces': None, 'extraInfo': '95 of the statements could not be deleted, due to use of blank nodes.'}
